# HW04: EPH Experiments — Stiffness, Damping, and Equifinality
**Computational Sensorimotor Control — Week 4**

Three experiments that probe the properties of λ control:
1. **Stiffness** — C-command (co-contraction) modulates endpoint stiffness
2. **Damping** — the μ parameter provides velocity-dependent stability
3. **Equifinality** — spring-like convergence under perturbation, and where it breaks down


In [ ]:
!pip3 install git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

from smc import (
    Arm, make_muscles, simulate_lambda, Q_REF, L1, L2,
    mass_matrix, joint_accelerations, rk4_step, C_EXP, MU_LAMBDA,
)

arm = Arm()
start_hand = arm.forward_kinematics(Q_REF)

def lambda_for_posture(q_target, C=0.25):
    """Compute λ values that hold the arm at q_target with co-contraction C.
    
    The C-command is a single scalar in RADIANS — the CNS sends one number,
    and each muscle's threshold shifts proportionally to its total moment arm:
        Δλ_j = (|r_sh_j| + |r_el_j|) × C
    
    Biarticular muscles (e.g., tri_lg with |r_sh|+|r_el| = 0.06 m/rad) get
    larger threshold shifts than monoarticular muscles (e.g., tri_l with 0.02 m/rad).
    
    Parameters
    ----------
    q_target : array, shape (2,)
        Target joint angles (rad).
    C : float
        C-command in radians. Controls co-contraction stiffness.
        C = 0.25 rad is a moderate default.
    
    Returns
    -------
    lambdas : array, shape (6,)
        Threshold values for each muscle (meters).
    """
    muscles = make_muscles()
    lams = np.zeros(6)
    for j, m in enumerate(muscles):
        ml = m.length(q_target)
        lams[j] = ml - (abs(m.r_sh) + abs(m.r_el)) * C
    return lams
def make_ramp(lam_init, lam_final, t_start=0.05, duration=0.35):
    def fn(t):
        if t < t_start: return lam_init.copy()
        elif t < t_start + duration:
            return lam_init + (t - t_start) / duration * (lam_final - lam_init)
        else: return lam_final.copy()
    return fn

def ik(x, y): return arm.inverse_kinematics(x, y)

# Standard reach target
TARGET_XY = np.array([start_hand[0] + 0.08, start_hand[1]])
TARGET_Q = ik(TARGET_XY[0], TARGET_XY[1])

print(f"Start: ({start_hand[0]*100:.1f}, {start_hand[1]*100:.1f}) cm")
print(f"Target: ({TARGET_XY[0]*100:.1f}, {TARGET_XY[1]*100:.1f}) cm")
print("Setup complete ✓")

---
## Part 1: Stiffness Control via C-Command (30 pts)

### Task 1.1 — Vary Co-Contraction Level (15 pts)


In [ ]:
c_levels = [0.05, 0.10, 0.25, 0.35, 0.50]  # C-command in radians
colors = ['#E74C3C', '#E67E22', '#27AE60', '#2E86AB', '#8e44ad']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
results_p1 = {}

for cc, col in zip(c_levels, colors):
    li = lambda_for_posture(Q_REF, cc)
    lf = lambda_for_posture(TARGET_Q, cc)
    t, s, h, _ = simulate_lambda(make_ramp(li, lf), T=1.5)
    
    d = np.linalg.norm(h - start_hand, axis=1) * 100
    v = np.zeros(len(t)); v[1:] = np.linalg.norm(np.diff(h, axis=0)/0.0001, axis=1) * 100
    
    # ──── TODO: COMPUTE SETTLING TIME (first time after 400ms when speed < 1 cm/s) ────
    # Hint: loop from index int(0.4/0.0001) onward, check if v[k] < 1.0
    # and v stays below 1.0 for the next 1000 samples. Store the time in ms.
    settle_idx = None
    for k in range(int(0.4/0.0001), len(v)):
        if ...:   # TODO: what condition means "settled"?
            settle_idx = k; break
    settle_t = t[settle_idx]*1000 if settle_idx else np.nan
    
    # ──── TODO: COMPUTE PEAK OVERSHOOT ────
    # Overshoot = maximum displacement minus final displacement
    overshoot = ...   # TODO
    # ──────────────────────────────────────
    
    results_p1[cc] = {'t': t, 'd': d, 'v': v, 'settle': settle_t, 'overshoot': overshoot}
    ax1.plot(t*1000, d, color=col, lw=2, label=f'C={cc:.2f} rad')
    ax2.plot(t*1000, v, color=col, lw=2, label=f'C={cc:.2f} rad')

ax1.set_ylabel('Displacement (cm)'); ax1.legend(fontsize=10)
ax1.set_title('Task 1.1: Effect of Co-Contraction on Reaching')
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Hand Speed (cm/s)')
ax2.axhline(1.0, color='gray', ls=':', alpha=0.5); ax2.legend(fontsize=10)
plt.tight_layout(); plt.show()

# Summary table
print(f"{'C (rad)':>8} {'Settle (ms)':>12} {'Overshoot (cm)':>15}")
print("-" * 38)
for cc in c_levels:
    r = results_p1[cc]
    print(f"{cc:>8} {r['settle']:>12.0f} {r['overshoot']:>15.2f}")

In [ ]:
# Plot settling time and overshoot vs C-command
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ccs = c_levels
settles = [results_p1[cc]['settle'] for cc in ccs]
overshoots = [results_p1[cc]['overshoot'] for cc in ccs]

ax1.plot(ccs, settles, 'o-', color='#2E86AB', lw=2, ms=8)
ax1.set_xlabel('C-Command (rad)'); ax1.set_ylabel('Settling Time (ms)')
ax1.set_title('Settling Time vs. C-Command')

ax2.plot(ccs, overshoots, 's-', color='#E74C3C', lw=2, ms=8)
ax2.set_xlabel('C-Command (rad)'); ax2.set_ylabel('Peak Overshoot (cm)')
ax2.set_title('Overshoot vs. C-Command')

plt.tight_layout(); plt.show()

### Task 1.2 — Endpoint Stiffness Test (10 pts)


In [ ]:
# After each reach settles (t=1.0s), apply a brief test perturbation
# and measure the resulting deflection → proxy for endpoint stiffness

def measure_stiffness(C_val, pert_mag=1.0, pert_dur=0.02):
    """Reach, settle, then apply a test perturbation. Return deflection."""
    li = lambda_for_posture(Q_REF, C_val)
    lf = lambda_for_posture(TARGET_Q, C_val)
    lam_fn = make_ramp(li, lf)
    
    # ──── TODO: WRITE THE PERTURBATION FUNCTION ────
    # It should apply pert_mag N·m at the shoulder between t=1.0 and t=1.0+pert_dur
    # Return np.zeros(2) at all other times
    def pert(t):
        ...   # TODO: return np.array([pert_mag, 0.0]) when? np.zeros(2) otherwise?
    # ──────────────────────────────────────
    
    # Unperturbed baseline
    t_np, _, h_np, _ = simulate_lambda(lam_fn, T=1.3)
    # Perturbed
    t_p, _, h_p, _ = simulate_lambda(lam_fn, T=1.3, perturb_fn=pert)
    
    # ──── TODO: COMPUTE MAX DEFLECTION ────
    # Compare perturbed vs unperturbed hand positions AFTER t=1.0s
    idx_start = int(1.0 / 0.0001)
    deflection = ...   # TODO: max distance between h_p and h_np after idx_start, in cm
    # ──────────────────────────────────────
    
    return deflection

deflections = [measure_stiffness(cc) for cc in c_levels]

# Plotting (provided)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(c_levels, deflections, 'D-', color='#27AE60', lw=2, ms=10)
ax.set_xlabel('C-Command (rad)', fontsize=13)
ax.set_ylabel('Test Deflection (cm)', fontsize=13)
ax.set_title('Endpoint Stiffness: Higher C-Command → Less Deflection', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"{'C (rad)':>8} {'Deflection (cm)':>16}")
print("-" * 26)
for cc, d in zip(c_levels, deflections):
    print(f"{cc:>8} {d:>16.3f}")

*Write your answer here.*


---
## Part 1b: Speed Control via λ Ramp Shape (20 pts)

In the lab, we drove reaching with **constant-rate (linear)** λ ramps. But are the resulting velocity profiles truly bell-shaped and symmetric like the data Morasso (1981) observed?

This task has two stages:
1. Try linear ramps at different durations → measure velocity asymmetry
2. Replace the linear ramp with a **sigmoidal** ramp → see if symmetry improves

### Task 1b.1 — Linear Ramps at Different Durations (8 pts)

Perform a straight-ahead reach of **20 cm** from Q_REF at five ramp durations: 600, 700, 800, 900, and 1000 ms. Use C = 0.25 rad. Ramp onset at t = 50 ms.

For each, measure peak hand speed, movement duration, and **velocity symmetry** (time of peak speed / movement duration — 0.5 = perfectly symmetric).


In [ ]:
# 20 cm straight ahead
target_20 = np.array([start_hand[0] + 0.20, start_hand[1]])
target_q_20 = ik(target_20[0], target_20[1])

ramp_durations = [0.60, 0.70, 0.80, 0.90, 1.00]  # seconds
ramp_colors = ['#E74C3C', '#E67E22', '#27AE60', '#2E86AB', '#8e44ad']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
results_linear = {}

for dur, col in zip(ramp_durations, ramp_colors):
    li = lambda_for_posture(Q_REF, 0.25)
    lf = lambda_for_posture(target_q_20, 0.25)
    
    # ──── TODO: CREATE THE λ RAMP WITH THE CORRECT DURATION ────
    lam_fn = make_ramp(li, lf, t_start=0.05, duration=...)  # TODO
    # ──────────────────────────────────────
    
    t, s, h, _ = simulate_lambda(lam_fn, T=2.5)
    d = np.linalg.norm(h - start_hand, axis=1) * 100
    v = np.zeros(len(t)); v[1:] = np.linalg.norm(np.diff(h, axis=0) / 0.0001, axis=1) * 100
    
    # ──── TODO: COMPUTE PEAK SPEED, MOVEMENT DURATION, SYMMETRY ────
    peak_speed = ...   # TODO: max of v
    peak_time = ...    # TODO: time of peak in ms
    
    above = np.where(v > 1.0)[0]
    if len(above) > 0:
        mt_start = ...   # TODO: time of first sample > 1 cm/s (ms)
        mt_end = ...     # TODO: time of last sample > 1 cm/s (ms)
        move_dur = mt_end - mt_start
    else:
        mt_start = mt_end = move_dur = np.nan
    
    symmetry = ...   # TODO: (peak_time - mt_start) / move_dur
    # ──────────────────────────────────────
    
    results_linear[dur] = {
        'peak_speed': peak_speed, 'peak_time': peak_time,
        'move_dur': move_dur, 'symmetry': symmetry,
    }
    
    ax1.plot(t*1000, d, color=col, lw=2, label=f'ramp={int(dur*1000)} ms')
    ax2.plot(t*1000, v, color=col, lw=2, label=f'ramp={int(dur*1000)} ms')

ax1.axhline(20.0, color='gray', ls=':', alpha=0.5)
ax1.set_ylabel('Displacement (cm)'); ax1.legend(fontsize=10)
ax1.set_title('Task 1b.1: Linear λ Ramps — Different Durations (20 cm reach)')
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Hand Speed (cm/s)')
ax2.axhline(1.0, color='gray', ls=':', alpha=0.3); ax2.legend(fontsize=10)
ax2.set_xlim(0, 2000)
plt.tight_layout(); plt.show()

print(f"{'Ramp (ms)':>10} {'Peak (cm/s)':>12} {'Move Dur (ms)':>14} {'Symmetry':>10}")
print("-" * 48)
for dur in ramp_durations:
    r = results_linear[dur]
    print(f"{int(dur*1000):>10} {r['peak_speed']:>12.1f} {r['move_dur']:>14.0f} {r['symmetry']:>10.2f}")
print(f"\nPerfectly symmetric = 0.50. Values < 0.50 mean peak is early.")

### Task 1b.2 — Sigmoidal λ Ramps (7 pts)

The velocity profiles above are asymmetric because a linear ramp changes λ at a constant rate — but movement onset and offset require gradual acceleration and deceleration.

What if λ itself follows an **S-shaped** trajectory — slow start, fast middle, slow end? The minimum-jerk temporal profile is a natural choice:

$$\lambda(t) = \lambda_{start} + \Delta\lambda \cdot (10\tau^3 - 15\tau^4 + 6\tau^5), \quad \tau = \frac{t - t_{start}}{\text{duration}}$$

Implement `make_sigmoid_ramp` and repeat the same 5 reaches. Compare the symmetry ratios.


In [ ]:
# ──── TODO: IMPLEMENT make_sigmoid_ramp ────
# This function should shape λ using the minimum-jerk temporal profile:
#   s(τ) = 10τ³ − 15τ⁴ + 6τ⁵,  where τ = (t − t_start) / duration
# Before t_start: return lam_init
# During ramp: return lam_init + s(τ) · (lam_final − lam_init)
# After ramp: return lam_final

def make_sigmoid_ramp(lam_init, lam_final, t_start=0.05, duration=0.35):
    """Sigmoidal (minimum-jerk shaped) λ ramp."""
    def fn(t):
        if t < t_start:
            return lam_init.copy()
        elif t < t_start + duration:
            tau = (t - t_start) / duration
            s = ...   # TODO: the minimum-jerk polynomial in τ
            return lam_init + s * (lam_final - lam_init)
        else:
            return lam_final.copy()
    return fn
# ──────────────────────────────────────

# Same 5 reaches, now with sigmoidal ramps
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
results_sigmoid = {}

for dur, col in zip(ramp_durations, ramp_colors):
    li = lambda_for_posture(Q_REF, 0.25)
    lf = lambda_for_posture(target_q_20, 0.25)
    lam_fn = make_sigmoid_ramp(li, lf, t_start=0.05, duration=dur)
    
    t, s, h, _ = simulate_lambda(lam_fn, T=2.5)
    d = np.linalg.norm(h - start_hand, axis=1) * 100
    v = np.zeros(len(t)); v[1:] = np.linalg.norm(np.diff(h, axis=0) / 0.0001, axis=1) * 100
    
    peak_speed = v.max()
    peak_time = t[np.argmax(v)] * 1000
    above = np.where(v > 1.0)[0]
    if len(above) > 0:
        mt_start = t[above[0]] * 1000; mt_end = t[above[-1]] * 1000
        move_dur = mt_end - mt_start
    else:
        mt_start = mt_end = move_dur = np.nan
    symmetry = (peak_time - mt_start) / move_dur if move_dur > 0 else np.nan
    
    results_sigmoid[dur] = {
        'peak_speed': peak_speed, 'peak_time': peak_time,
        'move_dur': move_dur, 'symmetry': symmetry,
    }
    
    ax1.plot(t*1000, d, color=col, lw=2, label=f'ramp={int(dur*1000)} ms')
    ax2.plot(t*1000, v, color=col, lw=2, label=f'ramp={int(dur*1000)} ms')

ax1.axhline(20.0, color='gray', ls=':', alpha=0.5)
ax1.set_ylabel('Displacement (cm)'); ax1.legend(fontsize=10)
ax1.set_title('Task 1b.2: Sigmoidal λ Ramps — Same Durations')
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Hand Speed (cm/s)')
ax2.axhline(1.0, color='gray', ls=':', alpha=0.3); ax2.legend(fontsize=10)
ax2.set_xlim(0, 2000)
plt.tight_layout(); plt.show()

print(f"{'Ramp (ms)':>10} {'Peak (cm/s)':>12} {'Move Dur (ms)':>14} {'Symmetry':>10}")
print("-" * 48)
for dur in ramp_durations:
    r = results_sigmoid[dur]
    print(f"{int(dur*1000):>10} {r['peak_speed']:>12.1f} {r['move_dur']:>14.0f} {r['symmetry']:>10.2f}")

In [ ]:
# Direct comparison: linear vs sigmoidal symmetry
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

durs_ms = [int(d*1000) for d in ramp_durations]
sym_lin = [results_linear[d]['symmetry'] for d in ramp_durations]
sym_sig = [results_sigmoid[d]['symmetry'] for d in ramp_durations]
pk_lin = [results_linear[d]['peak_speed'] for d in ramp_durations]
pk_sig = [results_sigmoid[d]['peak_speed'] for d in ramp_durations]

ax1.plot(durs_ms, sym_lin, 'o-', color='#E74C3C', lw=2, ms=8, label='Linear ramp')
ax1.plot(durs_ms, sym_sig, 's-', color='#27AE60', lw=2, ms=8, label='Sigmoidal ramp')
ax1.axhline(0.5, color='gray', ls=':', alpha=0.5, label='Perfect symmetry')
ax1.set_xlabel('Ramp Duration (ms)', fontsize=13)
ax1.set_ylabel('Symmetry Ratio', fontsize=13)
ax1.set_title('Velocity Profile Symmetry', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)

ax2.plot(durs_ms, pk_lin, 'o-', color='#E74C3C', lw=2, ms=8, label='Linear ramp')
ax2.plot(durs_ms, pk_sig, 's-', color='#27AE60', lw=2, ms=8, label='Sigmoidal ramp')
ax2.set_xlabel('Ramp Duration (ms)', fontsize=13)
ax2.set_ylabel('Peak Speed (cm/s)', fontsize=13)
ax2.set_title('Peak Hand Speed', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)

plt.tight_layout(); plt.show()

print("\nSigmoidal ramps → symmetry closer to 0.50 (more bell-shaped)")
print("Linear ramps → asymmetric (long deceleration tail)")

### Question 1b.1 (5 pts)

**(a)** How does the symmetry ratio differ between linear and sigmoidal ramps? Which produces velocity profiles closer to the symmetric bell shape observed by Morasso (1981)?

**(b)** The sigmoidal ramp uses the minimum-jerk temporal profile (10τ³ − 15τ⁴ + 6τ⁵) to shape the λ trajectory. What does this imply about the "simplicity" of EPH? If the brain needs to shape λ ramps with a minimum-jerk profile to produce kinematically correct movements, has the λ model really avoided trajectory planning — or has it simply hidden the planner inside the control signal?

**(c)** Compare the peak speeds between linear and sigmoidal ramps at the same duration. Are they the same? If different, explain why in terms of the rate of λ change at movement midpoint.


*Write your answer here.*


---
## Part 2: The Damping Parameter μ (30 pts)

### Task 2.1 — Vary μ from 0 to 0.12 (20 pts)

You need to write a custom simulation loop that uses a different μ value. The smc library defaults to MU_LAMBDA = 0.06.


In [ ]:
# Custom simulator with adjustable μ
from smc import make_muscles as _mk, rk4_step as _rk4, joint_accelerations as _ja, C_EXP
from smc.muscle import force_velocity_multiplier

def sim_lambda_custom_mu(lam_fn, mu, T=1.0, dt=0.0001):
    """Simulate λ control with a custom velocity sensitivity μ."""
    muscles = _mk()
    state = np.concatenate([Q_REF, [0., 0.]])
    t = np.arange(0, T, dt); n = len(t)
    hand = np.zeros((n, 2)); states = np.zeros((n, 4))
    
    for i in range(n):
        q, qd = state[:2], state[2:]
        hand[i] = arm.forward_kinematics(q); states[i] = state
        lams = lam_fn(t[i]); tau = np.zeros(2)
        for j, m in enumerate(muscles):
            ml = m.length(q); mv = m.velocity(qd)
            
            # ──── TODO: COMPUTE ACTIVATION WITH CUSTOM μ ────
            # The standard law is: A = [l - λ + μ·dl/dt]⁺
            # Use the 'mu' parameter passed to this function (NOT MU_LAMBDA)
            a_m = max(0.0, ...)   # TODO: fill in the threshold law using ml, lams[j], mu, mv
            # ────────────────────────────────────────────────
            
            a_mm = a_m * 1000.0   # convert meters → mm for exponential
            mt = m.rho * (np.exp(C_EXP * a_mm) - 1)
            # Calcium dynamics (same as Week 2)
            m.ca += dt * np.array([m.ca[1], (mt - m.ca[0] - 2*0.015*m.ca[1]) / 0.015**2])
            m.ca[0] = max(m.ca[0], 0)
            f = m.ca[0] * force_velocity_multiplier(mv) + m.k * (ml - m.rl)
            tau[0] += m.r_sh * f; tau[1] += m.r_el * f
        state = _rk4(state, lambda s: np.array([s[2],s[3],*_ja(s[:2],s[2:],tau,0.)]), dt)
    return t, states, hand

# Test across μ values
mu_values = [0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.12]
mu_colors = ['#E74C3C','#E67E22','#F1C40F','#27AE60','#2E86AB','#3498db','#8e44ad']

li = lambda_for_posture(Q_REF, 0.25)
lf = lambda_for_posture(TARGET_Q, 0.25)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
results_p2 = {}

for mu, col in zip(mu_values, mu_colors):
    t, s, h = sim_lambda_custom_mu(make_ramp(li, lf), mu=mu, T=1.5)
    d = np.linalg.norm(h - start_hand, axis=1) * 100
    v = np.zeros(len(t)); v[1:] = np.linalg.norm(np.diff(h, axis=0)/0.0001, axis=1) * 100
    
    # ──── TODO: MEASURE SETTLING TIME ────
    # (same logic as Part 1.1 — reuse your approach)
    settle_idx = None
    for k in range(int(0.4/0.0001), len(v)):
        if ...:   # TODO
            settle_idx = k; break
    settle_t = t[settle_idx]*1000 if settle_idx else np.nan
    # ──────────────────────────────────────
    
    # ──── TODO: DETECT OSCILLATION ────
    # Count direction reversals in displacement after the ramp ends
    # If there are ≥3 reversals, the arm is oscillating
    d_post = d[int(0.4/0.0001):]
    sign_changes = ...   # TODO: count how many times the direction of d changes
    oscillates = sign_changes >= 3
    # ──────────────────────────────────
    
    overshoot = d.max() - d[-1]
    results_p2[mu] = {'settle': settle_t, 'overshoot': overshoot, 'osc': oscillates}
    
    ax1.plot(t*1000, d, color=col, lw=2, label=f'μ={mu:.2f}')
    ax2.plot(t*1000, v, color=col, lw=2, label=f'μ={mu:.2f}')

ax1.set_ylabel('Displacement (cm)'); ax1.legend(fontsize=9)
ax1.set_title('Task 2.1: Effect of Velocity Sensitivity μ')
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Hand Speed (cm/s)')
ax2.axhline(1.0, color='gray', ls=':', alpha=0.5); ax2.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f"{'μ':>6} {'Settle (ms)':>12} {'Overshoot (cm)':>15} {'Oscillates?':>12}")
print("-" * 48)
for mu in mu_values:
    r = results_p2[mu]
    print(f"{mu:>6.2f} {r['settle']:>12.0f} {r['overshoot']:>15.2f} {'YES' if r['osc'] else 'no':>12}")

### Task 2.2 — Find the Critical μ (5 pts)


In [ ]:
# ──── TODO: FIND THE CRITICAL μ ────
# Sweep μ from 0 to 0.06 in steps of 0.005
# For each, simulate and count displacement reversals after the ramp
# Report the approximate μ below which the arm oscillates (≥3 reversals)

for mu_test in np.arange(0, 0.06, 0.005):
    t, s, h = sim_lambda_custom_mu(make_ramp(li, lf), mu=mu_test, T=1.5)
    d = np.linalg.norm(h - start_hand, axis=1) * 100
    
    # TODO: count sign changes in displacement after ramp (same as Task 2.1)
    sign_changes = ...
    osc = "OSCILLATES" if sign_changes >= 3 else "settles"
    print(f"  μ = {mu_test:.3f}: {osc} ({sign_changes} reversals)")

# TODO: report the critical μ value
# print(f"\nCritical μ ≈ ??? s")

*Write your answer here.*


---
## Part 3: Equifinality Under Stress (20 pts)

### Task 3.1 — Perturbation Magnitude Sweep (15 pts)


In [ ]:
pert_mags = [0.5, 1, 2, 5, 10, 15, 20]

li = lambda_for_posture(Q_REF, 0.25)
lf = lambda_for_posture(TARGET_Q, 0.25)
lam_fn = make_ramp(li, lf)

# Unperturbed reference
t_np, _, h_np, _ = simulate_lambda(lam_fn, T=1.5)

peak_devs = []
endpoint_errors = []

for mag in pert_mags:
    # ──── TODO: WRITE THE PERTURBATION FUNCTION ────
    # Apply 'mag' N·m at the shoulder between t=0.25 and t=0.30
    def pert(t, m=mag):
        ...   # TODO: return np.array([m, 0.0]) when? np.zeros(2) otherwise?
    # ──────────────────────────────────────
    
    _, s_p, h_p, _ = simulate_lambda(lam_fn, T=1.5, perturb_fn=pert)
    
    # ──── TODO: COMPUTE PEAK DEVIATION AND ENDPOINT ERROR ────
    dev = np.linalg.norm(h_p - h_np, axis=1) * 100
    peak_devs.append(...)       # TODO: maximum of dev
    endpoint_errors.append(...) # TODO: distance between final positions of h_p and h_np (in cm)
    # ──────────────────────────────────────

# Plotting (provided)
fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

ax1.plot(pert_mags, peak_devs, 'o-', color='#E74C3C', lw=2, ms=8, label='Peak path deviation')
ax2.plot(pert_mags, endpoint_errors, 's-', color='#2E86AB', lw=2, ms=8, label='Endpoint error')

ax1.set_xlabel('Perturbation Magnitude (N·m)', fontsize=13)
ax1.set_ylabel('Peak Path Deviation (cm)', fontsize=13, color='#E74C3C')
ax2.set_ylabel('Endpoint Error (cm)', fontsize=13, color='#2E86AB')
ax1.set_title('Equifinality Under Increasing Perturbation', fontsize=14, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=11)

plt.tight_layout(); plt.show()

print(f"{'Pert (N·m)':>10} {'Peak Dev (cm)':>14} {'Endpoint Err (cm)':>18}")
print("-" * 44)
for mag, pd, ee in zip(pert_mags, peak_devs, endpoint_errors):
    flag = " ← BREAKDOWN" if ee > 1.0 else ""
    print(f"{mag:>10.1f} {pd:>14.2f} {ee:>18.3f}{flag}")

*Write your answer here.*


---
## Part 4: Write-Up (20 pts)

Write your Methods & Results section here (300–500 words). Include:
- **Methods:** Model description (cite Gribble et al. 1998, Feldman 1966), three experiments, parameter ranges.
- **Results:** Key findings from Parts 1–3. Reference your figures. State the critical μ value and the perturbation magnitude where equifinality breaks down.
- **Connection to HW03:** How does the λ model resolve Model B instability? What trade-offs remain?

*Write your answer here.*
